In [11]:
import pandas as pd

# =========================
# Configuration
# =========================
INPUT_CSV = "/content/raw_data.csv"

REPLAN_CSV = "/content/raw_data_All_Success.csv"

COL_MAP = "Map"
COL_ALG = "Algorithm"
COL_P   = "Desired Safe prob"
COL_RT  = "Runtime"

COL_REPLAN = "Number Of Replans"
COL_ONLINE_SST = "Online Sum of Service Time"

COL_RUNTIME_REP = "Runtime"

# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_CSV)
df_rep = pd.read_csv(REPLAN_CSV)

def is_valid_runtime(x) -> bool:
    if pd.isna(x):
        return False
    s = str(x).strip()
    return s != "" and s.upper() != "NONE"

# =========================
# df: used for Success rate (over all instances)
# =========================
df["_valid_runtime"] = df[COL_RT].apply(is_valid_runtime)
df["_p_num"] = pd.to_numeric(df[COL_P], errors="coerce")

def build_algo_label(row) -> str:
    alg = str(row[COL_ALG]).strip()
    p_txt = str(row[COL_P]).strip()
    if alg == "CBSSsst":
        return "CBSSsst"
    return f"{alg} p={p_txt}"

df["_algo_label"] = df.apply(build_algo_label, axis=1)

# Order: CBSSsst -> RCbssTS -> RCbssTA
GROUP_ORDER = {"CBSSsst": 0, "RCbssTS": 1, "RCbssTA": 2}
df["_group"] = df[COL_ALG].astype(str).str.strip().map(GROUP_ORDER).fillna(99).astype(int)

# p sorting for TS/TA (CBSSsst gets inf)
df["_p_for_sort"] = df["_p_num"]
df.loc[df["_group"] == 0, "_p_for_sort"] = float("inf")

# =========================
# df_rep (All_Success): normalize + compute:
# - replan>=1 count
# - mean online SST
# - mean runtime (NOT online)
# =========================
df_rep[COL_ALG] = df_rep[COL_ALG].astype(str).str.strip()
df_rep[COL_MAP] = df_rep[COL_MAP].astype(str)

df_rep["_p_num"] = pd.to_numeric(df_rep[COL_P], errors="coerce")

def build_algo_label_rep(row) -> str:
    alg = str(row[COL_ALG]).strip()
    p_txt = str(row[COL_P]).strip()
    if alg == "CBSSsst":
        return "CBSSsst"
    return f"{alg} p={p_txt}"

df_rep["_algo_label"] = df_rep.apply(build_algo_label_rep, axis=1)

df_rep["_replan_num"] = pd.to_numeric(df_rep[COL_REPLAN], errors="coerce").fillna(0)
df_rep["_online_sst"] = pd.to_numeric(df_rep[COL_ONLINE_SST], errors="coerce")

df_rep["_valid_runtime"] = df_rep[COL_RUNTIME_REP].apply(is_valid_runtime)
df_rep["_runtime_num"] = pd.to_numeric(df_rep[COL_RUNTIME_REP], errors="coerce")
df_rep.loc[~df_rep["_valid_runtime"], "_runtime_num"] = pd.NA

rep_summary = (
    df_rep.groupby([COL_MAP, "_algo_label"], dropna=False)
          .agg(
              replan_ge_1_count=("_replan_num", lambda s: int((s >= 1).sum())),
              online_sst_mean=("_online_sst", "mean"),
              avg_runtime_mean=("_runtime_num", "mean"),
          )
          .reset_index()
)

# =========================
# Print table per map + separators
# + add: replan>=1 count + mean online SST + mean runtime (from df_rep)
# =========================
for map_name, g in df.groupby(COL_MAP, dropna=False):
    summary = (
        g.groupby(["_algo_label", "_group", "_p_for_sort"], dropna=False)
         .agg(total=(COL_RT, "size"), success=("_valid_runtime", "sum"))
         .reset_index()
    )

    summary["Success rate"] = (summary["success"] / summary["total"] * 100).round(2)

    # merge replans + online sst + avg runtime from df_rep
    rep_map = rep_summary[rep_summary[COL_MAP].astype(str) == str(map_name)]
    summary = summary.merge(
        rep_map[[COL_MAP, "_algo_label", "replan_ge_1_count", "online_sst_mean", "avg_runtime_mean"]],
        how="left",
        on=["_algo_label"],
    )

    summary["replan_ge_1_count"] = summary["replan_ge_1_count"].fillna(0).astype(int)
    summary["online_sst_mean"] = summary["online_sst_mean"].round(2)
    summary["avg_runtime_mean"] = summary["avg_runtime_mean"].round(2)

    summary = summary.sort_values(
        by=["_group", "_p_for_sort", "_algo_label"],
        ascending=[True, True, True],
        na_position="last"
    ).reset_index(drop=True)

    col1 = "planner configuration"
    rows = [
        (
            r["_algo_label"],
            f"{r['Success rate']:.2f}",
            int(r["_group"]),
            int(r["replan_ge_1_count"]),
            "NA" if pd.isna(r["avg_runtime_mean"]) else f"{r['avg_runtime_mean']:.2f}",
            "NA" if pd.isna(r["online_sst_mean"]) else f"{r['online_sst_mean']:.2f}",
        )
        for _, r in summary.iterrows()
    ]

    width = max(len(col1), max((len(name) for name, *_ in rows), default=len(col1)))

    print("\n")
    print(f"Map: {map_name}")
    print("*" * 60)
    print(f"\n{col1.ljust(width)}  Success rate (%)  Replan>=1 (count)  Avg runtime (sec)  Avg online SST")
    print("=" * (width + 84))

    first_two_cut = 2 if len(rows) > 2 else len(rows)

    for i, (name, rate, grp, rep_cnt, rt_mean, sst_mean) in enumerate(rows):
        if i == first_two_cut and i < len(rows):
            print("-" * (width + 84))

        if grp == 2 and i > 0 and rows[i - 1][2] != 2:
            print("-" * (width + 84))

        print(
            f"{name.ljust(width)}  {rate.rjust(6)}           "
            f"{str(rep_cnt).rjust(6)}           "
            f"{str(rt_mean).rjust(14)}           "
            f"{str(sst_mean).rjust(12)}"
        )


/tmp/ipykernel_13833/4040955379.py:23: DtypeWarning: Columns (1,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_CSV)




Map: maze-32-32-2
************************************************************

planner configuration  Success rate (%)  Replan>=1 (count)  Avg runtime (sec)  Avg online SST
CBSSsst                 83.67               33                    16.25                 181.48
RCbssTS p=0             96.28               38                    16.16                 182.25
---------------------------------------------------------------------------------------------------------
RCbssTS p=0.05          80.33               13                    15.62                 180.88
RCbssTS p=0.25          79.72               13                    15.66                 180.82
RCbssTS p=0.5           78.89               12                    15.64                 180.95
RCbssTS p=0.8           76.00                1                    15.48                 180.81
RCbssTS p=0.95          73.00                0                    15.59                 180.96
RCbssTS p=0.99          70.50                0       

In [12]:
import pandas as pd
from scipy.stats import binomtest
from IPython.display import display, Markdown, HTML

# === Load data ===
df = pd.read_csv("raw_data.csv", low_memory=False)

# === Clean / define success ===
df["Desired Safe prob"] = pd.to_numeric(df["Desired Safe prob"], errors="coerce")
df["Success"] = df["Online Sum of Service Time"].notna()

map_col = "Map"
algorithms = ["RCbssTS", "RCbssTA"]

p_values = [0.05, 0.25, 0.5, 0.8, 0.95, 0.99, 0.999, 0.9999]

filtered = df[
    (df["Desired Safe prob"].isin(p_values)) &
    (df["Algorithm"].isin(algorithms))
].copy()


# === Columns that identify the same instance across algorithms ===
candidate_key_cols = [
    "Map",
    "Number of agents",
    "Number of goals",
    "Desired Safe prob",
    "Delay probability",
    "p_delay",
    "Instance",
    "Instance ID",
    "InstanceId",
    "Seed",
    "Run",
    "Problem",
    "File"
]

key_cols = [c for c in candidate_key_cols if c in filtered.columns]

# fallback: if your file has no explicit instance column, use all stable config-like columns
# but exclude outputs and algorithm
output_like_cols = [
    "Algorithm",
    "Success",
    "Online Sum of Service Time",
    "Runtime",
    "Solve Time",
    "Total Runtime",
    "Number of Replans",
    "Replan",
    "Replanning"
]

key_cols = [c for c in key_cols if c not in output_like_cols]


def exact_mcnemar_pvalue(group_df):
    """
    Exact McNemar test between RCbssTA and RCbssTS.
    Uses only matched instances that appear for both algorithms.
    """
    paired = group_df.pivot_table(
        index=key_cols,
        columns="Algorithm",
        values="Success",
        aggfunc="first"
    )

    if not {"RCbssTA", "RCbssTS"}.issubset(paired.columns):
        return float("nan")

    paired = paired.dropna(subset=["RCbssTA", "RCbssTS"])

    if paired.empty:
        return float("nan")

    ta = paired["RCbssTA"].astype(bool)
    ts = paired["RCbssTS"].astype(bool)

    # Discordant pairs
    b = ((ta == True) & (ts == False)).sum()   # TA succeeds, TS fails
    c = ((ta == False) & (ts == True)).sum()   # TA fails, TS succeeds

    n_discordant = b + c

    if n_discordant == 0:
        return 1.0

    return binomtest(
        k=min(b, c),
        n=n_discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue


def build_success_table(group_col):
    summary = (
        filtered
        .groupby([map_col, group_col, "Algorithm"], as_index=False)
        .agg(SuccessRate=("Success", "mean"))
    )

    summary["SuccessRate"] *= 100

    table = summary.pivot_table(
        index=[map_col, group_col],
        columns="Algorithm",
        values="SuccessRate"
    ).reset_index()

    table.columns.name = None

    # === McNemar p-value per row ===
    pvals = (
        filtered
        .groupby([map_col, group_col])
        .apply(exact_mcnemar_pvalue)
        .reset_index(name="McNemar p-value")
    )

    table = table.merge(pvals, on=[map_col, group_col], how="left")

    table = table.rename(columns={
        map_col: "Map",
        "RCbssTS": "RCbssTS success (%)",
        "RCbssTA": "RCbssTA success (%)"
    })

    table[["RCbssTS success (%)", "RCbssTA success (%)"]] = (
        table[["RCbssTS success (%)", "RCbssTA success (%)"]].round(2)
    )

    table["Significant"] = table["McNemar p-value"].apply(lambda p: "yes" if pd.notna(p) and p < 0.05 else "no")

    table["McNemar p-value"] = table["McNemar p-value"].apply(lambda p: "<0.0001" if pd.notna(p) and p < 0.0001 else round(p, 4))

    return table


# === Tables aggregated over all p values ===
by_goals = build_success_table("Number of goals")
by_agents = build_success_table("Number of agents")


# === Display two tables side by side per map ===
for map_name in sorted(filtered[map_col].unique()):
    display(Markdown(f"## {map_name}"))

    goals_table = (
        by_goals[by_goals["Map"] == map_name]
        .drop(columns=["Map"])
        .sort_values("Number of goals")
        .reset_index(drop=True)
    )

    agents_table = (
        by_agents[by_agents["Map"] == map_name]
        .drop(columns=["Map"])
        .sort_values("Number of agents")
        .reset_index(drop=True)
    )

    html = f"""
    <div style="display:flex; gap:40px; align-items:flex-start;">
        <div>
            <h3>Success rate by number of goals</h3>
            {goals_table.to_html(index=False)}
        </div>
        <div>
            <h3>Success rate by number of agents</h3>
            {agents_table.to_html(index=False)}
        </div>
    </div>
    """

    display(HTML(html))

/tmp/ipykernel_13833/646617874.py:120: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(exact_mcnemar_pvalue)
/tmp/ipykernel_13833/646617874.py:120: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(exact_mcnemar_pvalue)


## maze-32-32-2

Number of goals,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
15,97.73,92.90,<0.0001,yes
20,96.42,85.29,<0.0001,yes
25,94.31,41.25,<0.0001,yes
Number of agents,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
20,92.47,68.25,<0.0001,yes
30,95.97,75.56,<0.0001,yes
40,98.97,77.67,<0.0001,yes
50,97.19,71.11,<0.0001,yes


## random-32-32-20

Number of goals,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
15,97.46,96.29,<0.0001,yes
20,96.10,91.02,<0.0001,yes
25,95.85,58.94,<0.0001,yes
Number of agents,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
20,97.03,85.67,<0.0001,yes
30,96.83,88.06,<0.0001,yes
40,97.14,82.25,<0.0001,yes
50,94.89,72.36,<0.0001,yes


## room-32-32-4

Number of goals,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
15,96.35,91.25,<0.0001,yes
20,94.67,85.12,<0.0001,yes
25,90.62,50.88,<0.0001,yes
Number of agents,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
20,91.58,74.67,<0.0001,yes
30,91.86,75.03,<0.0001,yes
40,96.25,79.64,<0.0001,yes
50,95.83,73.67,<0.0001,yes


## warehouse-10-20-10-2-1

Number of goals,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
15,99.21,97.21,<0.0001,yes
20,97.44,93.17,<0.0001,yes
25,95.08,66.48,<0.0001,yes
Number of agents,RCbssTA success (%),RCbssTS success (%),McNemar p-value,Significant
20,96.78,88.31,<0.0001,yes
30,97.67,85.50,<0.0001,yes
40,97.22,86.81,<0.0001,yes
50,97.31,81.86,<0.0001,yes
